# ANN Orchestrator For Multi-Run Ensembling and Checkpointing

This notebook defines a reusable orchestrator that:
1. Configures a set of runs (architectures, hyperparameters, number of seeds).
2. Runs expanding-window forecasts and top-k ensembling, reporting both validation-loss and trailing-OOS ensembles.
3. Saves model checkpoints at each refit step (DeepSHAP-friendly: torch `state_dict` + preprocessing state).
4. Saves performance tuples per maturity: `(maturity, r2_oos, rsz_pval)`.

The example at the bottom runs a simple forward-only ANN with `n_models=10` and `k_top=1`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
import matplotlib.pyplot as plt
from utils.publication_lags import apply_fred_md_publication_lag
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array
import torch

repo_root = os.path.abspath('../..')

# Bianchi period:
start_date = '1971-08-31'
# end_date = '2018-12-31'
end_date = '2025-06-30' # kr and gsw end date

DATASET = 'kr' # or 'kr'
GAP = 11
MONTHLY = False
REALTIME = True

if MONTHLY:
    USE_MONTHLY_FORWARDS = True
else: 
    USE_MONTHLY_FORWARDS = False

maturities = [str(i) for i in range(12, 121) if i % 12 == 0] # select only yearly maturities

yields = bu.get_yields(type=DATASET, start=start_date, end=end_date, maturities=maturities) # type can be kr, lw, gsw
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna() # horizon=12 means holding for 12 months

# adjust fred_md start_date by 6 months to fetch enough data for shifting
fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=fred_md_start_date, end=end_date) # this is aligned to the last day of the previous month, so we get the same number of observations as the yields data

monthly_yields = bu.get_yields(type=DATASET, start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)]) # needed for monthly holding period excess returns. Not available for gsw
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna() # calculate monthly excess returns for robustness
monthly_forward = bu.get_monthly_forward_rates(monthly_yields)
# If wanted, apply per-series publication lag to latest-snapshot macro data
# from utils.publication_lags import apply_fred_md_publication_lag
# fred_md = apply_fred_md_publication_lag(fred_md_raw)  
# For results in paper, we shift all FRED-MD series by 1 month
# to reflect publication lag:
fred_md = fred_md_raw.shift(1)

# Drop TWEXAFEGSMTHx and ACOGNO as they start late
fred_md = fred_md.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'])
# Finally, revert fred_md to start_date, after transformations and lag adjustments
fred_md = fred_md[start_date:end_date]

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]
monthly_forward = monthly_forward.loc[monthly_forward.index <= xr.index[-1]]

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred_md, forward, yields)

if USE_MONTHLY_FORWARDS:
    forward = monthly_forward.copy()

X = pd.concat([fred_md, forward, yields],
               axis=1,
               keys=['fred', 'forward', 'yields'])

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
# For now, forward fill X
X = X.ffill()

groups = groups_as_array(X, level='group')
MATURITIES = ['24','36','48','60','84','120']

if MONTHLY:
    y_all = monthly_xr[MATURITIES].values
else:
    y_all = xr[MATURITIES].values

dates = xr.index

/home/ulrikts/Documents/NTNU/TIO4900-Replication/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:219: UserWarning: The following series are defined in get_fredmd_grouping() but are not present in the FRED-MD data: ['ACOGNO', 'TWEXAFEGSMTHx']. They may have been dropped or renamed.
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(


In [ ]:
def compute_top_k_ensemble(
    forecasts_array: np.ndarray,
    val_losses_array: np.ndarray,
    k: int,
    selection_metric: str = 'val_loss',
    y_true: np.ndarray | None = None,
    lookback: int = 120,
    min_history: int = 24,
    realization_lag: int = 0,
):
    T, n_seeds, n_outputs = forecasts_array.shape
    ensemble_forecast = np.full((T, n_outputs), np.nan)
    topk_indices = np.full((T, n_outputs, min(k, n_seeds)), -1, dtype=int)

    if selection_metric not in {'val_loss', 'trailing_oos'}:
        raise ValueError("selection_metric must be either 'val_loss' or 'trailing_oos'")

    if selection_metric == 'trailing_oos':
        if y_true is None:
            raise ValueError("y_true must be provided when selection_metric='trailing_oos'")
        y_true = np.asarray(y_true)
        if y_true.ndim == 1:
            y_true = y_true.reshape(-1, 1)
        if y_true.shape != (T, n_outputs):
            raise ValueError(f"y_true must have shape {(T, n_outputs)}; got {y_true.shape}")

    def _nanmean_with_counts(arr, axis):
        valid_counts = np.sum(~np.isnan(arr), axis=axis)
        sums = np.nansum(arr, axis=axis)
        means = np.divide(
            sums,
            valid_counts,
            out=np.full(np.shape(sums), np.nan, dtype=float),
            where=valid_counts > 0,
        )
        return means, valid_counts

    def _select_from_val_losses(t, m):
        v_losses = val_losses_array[t, :, m]
        valid_idx = np.where(~np.isnan(v_losses))[0]
        return v_losses, valid_idx

    for t in range(T):
        for m in range(n_outputs):
            if selection_metric == 'trailing_oos':
                hist_end = t - realization_lag
                hist_start = max(0, hist_end - lookback)
                if hist_end > hist_start:
                    trailing_err = (
                        forecasts_array[hist_start:hist_end, :, m]
                        - y_true[hist_start:hist_end, m][:, None]
                    ) ** 2
                    v_losses, counts = _nanmean_with_counts(trailing_err, axis=0)
                    valid_idx = np.where((~np.isnan(v_losses)) & (counts >= min_history))[0]
                else:
                    v_losses = None
                    valid_idx = np.array([], dtype=int)
            else:
                v_losses, valid_idx = _select_from_val_losses(t, m)

            if len(valid_idx) == 0:
                continue

            actual_k = min(k, len(valid_idx))
            sorted_valid_idx = valid_idx[np.argsort(v_losses[valid_idx])]
            selected = sorted_valid_idx[:actual_k]
            topk_indices[t, m, :actual_k] = selected

            selected_preds = forecasts_array[t, selected, m]
            valid_preds = selected_preds[~np.isnan(selected_preds)]
            if len(valid_preds) == 0:
                continue
            ensemble_forecast[t, m] = np.mean(valid_preds)

    return ensemble_forecast, topk_indices


def _score_ensemble(y_true, ensemble_forecast, maturities, benchmark='hist_mean', gap=0):
    r2s = wu.oos_r2(y_true, ensemble_forecast, benchmark=benchmark, gap=gap)
    pvals = np.array([
        bu.RSZ_Signif(y_true[:, i], ensemble_forecast[:, i], gap=gap)
        for i in range(y_true.shape[1])
    ])
    performance_tuples = list(zip(maturities, r2s.tolist(), pvals.tolist()))
    return r2s, pvals, performance_tuples


def _print_performance(title, performance_tuples):
    print(f"\n--- {title} ---")
    for mat, r2, pval in performance_tuples:
        print(f"  {mat}m: R2_OOS = {r2:.4f}, RSZ p-value = {pval:.3f}")


def _format_performance_rows(selection_metric, performance_tuples):
    return [
        {
            'selection_metric': selection_metric,
            'maturity': maturity,
            'r2_oos': r2,
            'rsz_pval': pval,
        }
        for maturity, r2, pval in performance_tuples
    ]


def _extract_scaler_state(scaler):
    if scaler is None:
        return None
    state = {}
    for attr in ['mean_', 'scale_', 'var_', 'n_samples_seen_', 'n_features_in_']:
        if hasattr(scaler, attr):
            val = getattr(scaler, attr)
            if isinstance(val, np.ndarray):
                state[attr] = val.copy()
            elif np.isscalar(val):
                state[attr] = val.item() if hasattr(val, 'item') else val
            else:
                state[attr] = val
    return state


def _extract_pca_state(pca):
    if pca is None:
        return None
    state = {}
    for attr in ['components_', 'mean_', 'explained_variance_', 'explained_variance_ratio_', 'n_components_']:
        if hasattr(pca, attr):
            val = getattr(pca, attr)
            state[attr] = val.copy() if isinstance(val, np.ndarray) else val
    return state


def _estimate_model_size_mb(wrapper_model: torch.nn.Module) -> float:
    n_params = sum(p.numel() for p in wrapper_model.parameters())
    return (n_params * 4) / (1024 ** 2)

In [6]:
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Callable
from tqdm import tqdm


@dataclass
class RunConfig:
    run_name: str
    model_builder: Callable[[int], object]
    n_models: int
    k_top: int
    maturities: list
    oos_start: pd.Timestamp
    gap: int = GAP
    refit_freq: int = 1
    benchmark: str = 'hist_mean'
    rsz_maxlags: int = 12
    progress: bool = False
    save_checkpoints: bool = True
    artifacts_root: Path = Path('../artifacts/orchestrator_runs')


def _save_checkpoint(wrapper, seed: int, t_index: int, date_value, run_dir: Path) -> Path:
    ckpt_dir = run_dir / 'checkpoints' / f'seed_{seed:03d}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f'step_{t_index:04d}_{pd.Timestamp(date_value).date()}.pt'

    x_scalers_macro_state = None
    if hasattr(wrapper, 'x_scalers_macro') and isinstance(wrapper.x_scalers_macro, dict):
        x_scalers_macro_state = {k: _extract_scaler_state(v) for k, v in wrapper.x_scalers_macro.items()}

    checkpoint = {
        'wrapper_class': wrapper.__class__.__name__,
        'wrapper_module': wrapper.__class__.__module__,
        'torch_state_dict': wrapper.model.state_dict() if hasattr(wrapper, 'model') and wrapper.model is not None else None,
        'best_params_': getattr(wrapper, 'best_params_', None),
        'fit_calls': getattr(wrapper, '_fit_calls', None),
        'x_scaler': _extract_scaler_state(getattr(wrapper, 'x_scaler', None)),
        'x_scaler_forward': _extract_scaler_state(getattr(wrapper, 'x_scaler_forward', None)),
        'x_scaler_fred': _extract_scaler_state(getattr(wrapper, 'x_scaler_fred', None)),
        'x_scalers_macro': x_scalers_macro_state,
        'y_scaler': _extract_scaler_state(getattr(wrapper, 'y_scaler', None)),
        'pca': _extract_pca_state(getattr(wrapper, 'pca', None)),
    }

    torch.save(checkpoint, ckpt_path)
    return ckpt_path


def run_experiment(cfg: RunConfig, X: pd.DataFrame, y_all: np.ndarray, dates: pd.DatetimeIndex, realtime=False):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = (cfg.artifacts_root / cfg.run_name / ts).resolve()
    run_dir.mkdir(parents=True, exist_ok=True)

    T = len(dates)
    n_outputs = y_all.shape[1] if y_all.ndim > 1 else 1

    all_forecasts = []
    all_val_losses = []
    ckpt_manifest = []

    model_iter = range(cfg.n_models)
    if cfg.progress:
        model_iter = tqdm(model_iter, desc='Seeds')

    for seed in model_iter:
        model = cfg.model_builder(seed)
        val_losses_for_seed = np.full((T, n_outputs), np.nan)

        # This callback is triggered at each refit step by expanding_window.
        def save_cb(model, refit_i, t_index, date_value, **kwargs):
            if hasattr(model, 'val_loss_') and model.val_loss_ is not None:
                val_losses_for_seed[t_index] = model.val_loss_
            if cfg.save_checkpoints:
                ckpt_path = _save_checkpoint(model, seed, t_index, date_value, run_dir)
                ckpt_manifest.append({
                    'seed': seed,
                    'refit_i': refit_i,
                    't_index': int(t_index),
                    'date': str(pd.Timestamp(date_value).date()),
                    'checkpoint_path': str(ckpt_path),
                })

        y_forecast = wu.expanding_window(
            model, X, y_all, dates, cfg.oos_start,
            gap=cfg.gap,
            refit_freq=cfg.refit_freq,
            save_callback=save_cb,
            progress=False,
            realtime=realtime,
            
        )

        all_forecasts.append(y_forecast)
        all_val_losses.append(val_losses_for_seed)

    forecasts_arr = np.stack(all_forecasts, axis=1)
    losses_arr = np.stack(all_val_losses, axis=1)

    val_forecast, val_topk_indices = compute_top_k_ensemble(
        forecasts_arr,
        losses_arr,
        cfg.k_top,
        selection_metric='val_loss',
    )
    trailing_oos_forecast, trailing_oos_topk_indices = compute_top_k_ensemble(
        forecasts_arr,
        losses_arr,
        cfg.k_top,
        selection_metric='trailing_oos',
        y_true=y_all,
        lookback=120,
        min_history=24,
        realization_lag=cfg.gap,
    )

    val_r2s, val_pvals, val_performance_tuples = _score_ensemble(
        y_all,
        val_forecast,
        cfg.maturities,
        benchmark=cfg.benchmark,
        gap=cfg.gap,
    )
    trailing_r2s, trailing_pvals, trailing_performance_tuples = _score_ensemble(
        y_all,
        trailing_oos_forecast,
        cfg.maturities,
        benchmark=cfg.benchmark,
        gap=cfg.gap,
    )

    _print_performance('Validation-loss ensemble', val_performance_tuples)
    _print_performance('Trailing OOS ensemble', trailing_performance_tuples)

    performance_df = pd.DataFrame(
        _format_performance_rows('val_loss', val_performance_tuples)
        + _format_performance_rows('trailing_oos', trailing_performance_tuples)
    )
    performance_df.to_csv(run_dir / 'performance.csv', index=False)

    # Persist arrays and metadata
    np.save(run_dir / 'forecasts_arr.npy', forecasts_arr)
    np.save(run_dir / 'losses_arr.npy', losses_arr)
    np.save(run_dir / 'ensemble_forecast_val_loss.npy', val_forecast)
    np.save(run_dir / 'ensemble_forecast_trailing_oos.npy', trailing_oos_forecast)
    np.save(run_dir / 'topk_indices_val_loss.npy', val_topk_indices)
    np.save(run_dir / 'topk_indices_trailing_oos.npy', trailing_oos_topk_indices)
    if cfg.save_checkpoints:
        pd.DataFrame(ckpt_manifest).to_csv(run_dir / 'checkpoint_manifest.csv', index=False)

    serializable_cfg = asdict(cfg)
    serializable_cfg['model_builder'] = str(cfg.model_builder)
    serializable_cfg['artifacts_root'] = str(cfg.artifacts_root)
    pd.Series(serializable_cfg).to_json(run_dir / 'run_config.json', indent=2)

    # Storage summary
    if cfg.save_checkpoints:
        ckpt_paths = list((run_dir / 'checkpoints').rglob('*.pt'))
        total_ckpt_bytes = sum(p.stat().st_size for p in ckpt_paths)
        num_checkpoints = len(ckpt_paths)
        total_checkpoint_gb = total_ckpt_bytes / (1024 ** 3)
    else:
        num_checkpoints = 0
        total_checkpoint_gb = 0.0

    summary = {
        'run_dir': str(run_dir),
        'num_checkpoints': num_checkpoints,
        'total_checkpoint_gb': total_checkpoint_gb,
        'save_checkpoints': cfg.save_checkpoints,
        'performance': {
            'val_loss': val_performance_tuples,
            'trailing_oos': trailing_performance_tuples,
        },
        'forecasts_arr_shape': forecasts_arr.shape,
        'losses_arr_shape': losses_arr.shape,
        'ensemble_shapes': {
            'val_loss': val_forecast.shape,
            'trailing_oos': trailing_oos_forecast.shape,
        },
    }

    return summary

# Main runner code

In [9]:
import csv
from models.ann_vector_validation import PyTorchMLPWrapper
from models.macro_forward_ann import MacroForwardANNWrapper
from models.group_ensemble_ann import GroupEnsembleANNWrapper

# Define experiment artifacts root with correct metadata in path
run_mode_str = "realtime" if REALTIME else "revised"
experiment_artifacts_root = Path(f'/home/ulrikts/Documents/NTNU/TIO4900-Replication/artifacts/orchestrator_runs/30_month_tuning_run_1971_2025_kr_ANNUAL_NONOVERLAPPING_{run_mode_str}')
experiment_artifacts_root.mkdir(parents=True, exist_ok=True)

def arch_to_name(arch):
    return 'direct' if len(arch) == 0 else '&'.join(str(x) for x in arch)

def base_run_config(run_name, model_builder, n_models=100, k_top=10, save_checkpoints=True):
    return RunConfig(
        run_name=run_name,
        model_builder=model_builder,
        n_models=n_models,
        k_top=k_top,
        maturities=MATURITIES,
        oos_start=pd.Timestamp('1990-01-31'),
        gap=GAP,
        refit_freq=1,
        benchmark='hist_mean',
        progress=True,
        save_checkpoints=save_checkpoints,
        artifacts_root=experiment_artifacts_root,
    )

def make_fwd_ann_cfg(arch, lr=0.01, epochs=1000, tune_every=60, activation='relu', param_grid=None, n_models=100, k_top=10, save_checkpoints=True):
    if param_grid is None:
        param_grid = {'penalty': [0.01, 0.001, 0.0001]}
    return base_run_config(
        run_name=f"fwd_ann_{arch_to_name(arch)}_{n_models}runs_top{k_top}",
        model_builder=lambda seed, arch=arch: PyTorchMLPWrapper(
            archi=arch,
            lr=lr,
            epochs=epochs,
            tune_every=tune_every,
            patience=50,
            param_grid=param_grid,
            seed=seed,
            activation=activation,
            use_pca=False,
            n_components=None,
            y_center=True,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )

def make_macro_forward_cfg(arch_macro, arch_forward=(3,), lr=0.01, epochs=1000, tune_every=60, activation='relu', param_grid=None, n_models=100, k_top=10, save_checkpoints=True):
    if param_grid is None:
        param_grid = {'penalty': [0.01, 0.001, 0.0001]}
    return base_run_config(
        run_name=(
            f"macro_fwd_ann_fwd{arch_to_name(arch_forward)}_"
            f"macro{arch_to_name(arch_macro)}_{n_models}runs_top{k_top}"
        ),
        model_builder=lambda seed, arch_macro=arch_macro, arch_forward=arch_forward: MacroForwardANNWrapper(
            archi_forward=arch_forward,
            archi_macro=arch_macro,
            lr=lr,
            epochs=epochs,
            tune_every=tune_every,
            patience=50,
            param_grid=param_grid,
            seed=seed,
            y_center=True,
            activation=activation,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )

def make_group_ensemble_cfg(arch_macro, arch_forward, lr=0.01, epochs=1000, tune_every=60, activation='relu', param_grid=None, n_models=100, k_top=10, save_checkpoints=True):
    if param_grid is None:
        param_grid = {
            'penalty': [0.01, 0.001, 0.0001],
            'dropout_rate': [0.0, 0.1, 0.3],
        }
    return base_run_config(
        run_name=(
            f"group_ens_ann_fwd{arch_to_name(arch_forward)}_"
            f"grp{arch_to_name(arch_macro)}_{n_models}runs_top{k_top}"
        ),
        model_builder=lambda seed, arch_macro=arch_macro, arch_forward=arch_forward: GroupEnsembleANNWrapper(
            archi_forward=arch_forward,
            archi_macro=arch_macro,
            lr=lr,
            epochs=epochs,
            tune_every=tune_every,
            patience=50,
            param_grid=param_grid,
            seed=seed,
            y_center=True,
            activation=activation,
        ),
        n_models=n_models,
        k_top=k_top,
        save_checkpoints=save_checkpoints,
    )

run_configs = []

# Monthly configs:
# run_configs = [make_fwd_ann_cfg((3,3), lr=0.001, epochs=1000, tune_every=60, activation='tanh', param_grid={'penalty':  [1e-4, 1e-3, 1e-2, 0.1]}, save_checkpoints=False, n_models=100)]
# run_configs += [make_macro_forward_cfg(arch_forward=(3,3,3), arch_macro=(32,), lr=0.001, epochs=1000, tune_every=60, activation='tanh', param_grid={'penalty':  [1e-4, 1e-3, 1e-2, 0.1], 'dropout_rate': [0.0, 0.1, 0.3]}, save_checkpoints=False, n_models=100)]
run_configs += [make_group_ensemble_cfg(arch_forward=(3,3), arch_macro=(1,), lr=0.001, epochs=1000, tune_every=30, activation='tanh', param_grid={'penalty':  [1e-4, 1e-3], 'dropout_rate': [0.0, 0.1, 0.3]}, save_checkpoints=True, n_models=100)]

master_summary_path = (experiment_artifacts_root / 'master_summary.csv').resolve()
master_summary_path.parent.mkdir(parents=True, exist_ok=True)

master_fields = [
    'timestamp', 'run_name', 'status', 'run_dir', 'num_checkpoints',
    'total_checkpoint_gb', 'forecasts_arr_shape', 'losses_arr_shape',
    'performance', 'error',
]

def append_master_summary(row_dict):
    write_header = not master_summary_path.exists()
    with open(master_summary_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=master_fields)
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)

all_summaries = []
failed_runs = []

for i, cfg in enumerate(run_configs, start=1):
    print(f"[{i}/{len(run_configs)}] Running {cfg.run_name} ...")
    ts_now = datetime.now().isoformat(timespec='seconds')

    try: 
        summary = run_experiment(cfg, X, y_all, dates, realtime=REALTIME)
        all_summaries.append(summary)

        append_master_summary({
            'timestamp': ts_now,
            'run_name': cfg.run_name,
            'status': 'ok',
            'run_dir': summary.get('run_dir', ''),
            'num_checkpoints': summary.get('num_checkpoints', ''),
            'total_checkpoint_gb': summary.get('total_checkpoint_gb', ''),
            'forecasts_arr_shape': summary.get('forecasts_arr_shape', ''),
            'losses_arr_shape': summary.get('losses_arr_shape', ''),
            'performance': summary.get('performance', ''),
            'error': '',
        })

    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        failed_runs.append({'run_name': cfg.run_name, 'error': err})
        print(f"FAILED {cfg.run_name} -> {err}")

        append_master_summary({
            'timestamp': ts_now,
            'run_name': cfg.run_name,
            'status': 'failed',
            'run_dir': '',
            'num_checkpoints': '',
            'total_checkpoint_gb': '',
            'forecasts_arr_shape': '',
            'losses_arr_shape': '',
            'performance': '',
            'error': err,
        })

print(f"Master summary: {master_summary_path}")

print(f"Completed {len(all_summaries)} successful runs out of {len(run_configs)} total.")

print(f"Failed runs: {len(failed_runs)}")

[1/1] Running group_ens_ann_fwd3&3_grp1_100runs_top10 ...


Seeds: 100%|██████████| 100/100 [4:49:48<00:00, 173.88s/it] 



--- Validation-loss ensemble ---
  24m: R2_OOS = -0.1695, RSZ p-value = 0.102
  36m: R2_OOS = -0.1860, RSZ p-value = 0.180
  48m: R2_OOS = -0.1364, RSZ p-value = 0.079
  60m: R2_OOS = -0.1081, RSZ p-value = 0.060
  84m: R2_OOS = -0.0760, RSZ p-value = 0.041
  120m: R2_OOS = 0.0002, RSZ p-value = 0.010

--- Trailing OOS ensemble ---
  24m: R2_OOS = -0.0635, RSZ p-value = 0.176
  36m: R2_OOS = -0.1485, RSZ p-value = 0.612
  48m: R2_OOS = -0.0320, RSZ p-value = 0.126
  60m: R2_OOS = -0.0690, RSZ p-value = 0.309
  84m: R2_OOS = -0.1113, RSZ p-value = 0.491
  120m: R2_OOS = -0.0304, RSZ p-value = 0.173
Master summary: /home/ulrikts/Documents/NTNU/TIO4900-Replication/artifacts/orchestrator_runs/30_month_tuning_run_1971_2025_kr_ANNUAL_NONOVERLAPPING_realtime/master_summary.csv
Completed 1 successful runs out of 1 total.
Failed runs: 0
